# ZoomVid: Trainable Video-Context Module on Frozen Qwen2.5-VL-3B

This notebook runs feature caching, baseline evaluation, and module training on a Google Colab T4 GPU.

## 1. Check GPU & Mount Google Drive

In [ ]:
# Verify GPU access (should show Tesla T4 ~15 GB VRAM)
!nvidia-smi

# Mount Google Drive to persist cache and checkpoints across sessions
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p "/content/drive/MyDrive/zoomvid/cache"
!mkdir -p "/content/drive/MyDrive/zoomvid/checkpoints"
!mkdir -p "/content/drive/MyDrive/zoomvid/eval_results"

## 2. Navigate to Project & Install Dependencies

In [ ]:
%cd /content/videomodelling

# Install dependencies and zoomvid in editable mode
!pip install -r zoomvid/requirements.txt
!pip install -e .

## 3. Run Unit Tests & VLM Equivalence Verification

In [ ]:
# Run unit test suite
!pytest zoomvid/tests/ -v

# Run full VLM equivalence and parameter freeze test
!python zoomvid/tests/test_vlm_colab_equivalence.py

## 4. Prepare Dataset Manifests & Feature Cache (Run Once)

In [ ]:
# Prepare train and dev split manifests (with strict benchmark de-duplication)
!python -m zoomvid.data.download_and_split --output-dir data/ --max-train 1000 --max-eval 200

## 5. Evaluate Reference Baselines (Uniform & Similarity)

In [ ]:
# 1. Evaluate Uniform frame sampling baseline
!python -m zoomvid.eval.attach_and_eval \
    --module-type uniform \
    --cache-dir /content/drive/MyDrive/zoomvid/cache \
    --eval-manifest data/dev_tempcompass_manifest.json \
    --budgets 1024 2048 4096 \
    --output-dir /content/drive/MyDrive/zoomvid/eval_results/uniform

# 2. Evaluate Question-Frame Similarity baseline
!python -m zoomvid.eval.attach_and_eval \
    --module-type similarity \
    --cache-dir /content/drive/MyDrive/zoomvid/cache \
    --eval-manifest data/dev_tempcompass_manifest.json \
    --budgets 1024 2048 4096 \
    --output-dir /content/drive/MyDrive/zoomvid/eval_results/similarity

## 6. Train DummyModule (Pipeline Smoke Test)

In [ ]:
!python -m zoomvid.train.trainer \
    --config zoomvid/configs/default_t4.yaml \
    --module-type dummy \
    --cache-dir /content/drive/MyDrive/zoomvid/cache \
    --train-manifest data/train_llava_manifest.json \
    --output-dir /content/drive/MyDrive/zoomvid/checkpoints/dummy_run \
    --max-steps 100

## 7. Plug in Your Custom Module & Train

Fill in the stubs in `zoomvid/module/zoomvid_module/` (`sketch_encoder.py`, `router.py`, `assembler.py`), then run:

In [ ]:
!python -m zoomvid.train.trainer \
    --config zoomvid/configs/default_t4.yaml \
    --module-type zoomvid \
    --cache-dir /content/drive/MyDrive/zoomvid/cache \
    --train-manifest data/train_llava_manifest.json \
    --output-dir /content/drive/MyDrive/zoomvid/checkpoints/zoomvid_run1 \
    --max-steps 1000

## 8. Evaluate Your Trained Checkpoint

In [ ]:
!python -m zoomvid.eval.attach_and_eval \
    --checkpoint /content/drive/MyDrive/zoomvid/checkpoints/zoomvid_run1/best_checkpoint.pt \
    --cache-dir /content/drive/MyDrive/zoomvid/cache \
    --eval-manifest data/dev_tempcompass_manifest.json \
    --budgets 1024 2048 4096 \
    --output-dir /content/drive/MyDrive/zoomvid/eval_results/zoomvid_run1